# Step C — SVD Polarity Resolution
Re-implements `C_SVD_all_15OBS.m`.  
Builds a symmetric sparse CC matrix per station, runs `svds(k=1)`,  
determines per-event polarity from the leading eigenvector,  
and applies the elbow-method threshold on `log₁₀|U₁|`.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import svds
from pathlib import Path

DATA  = Path('/Users/mczhang/Documents/GitHub/FM7/02-data')
OUT_B = DATA / 'B_CC_py'
OUT_C = DATA / 'C_SVD_py'; OUT_C.mkdir(exist_ok=True)
WAVE_H5 = DATA / 'A_ID' / 'waves_Y2_ge8.h5'
YEAR = 'Y2'

import h5py
with h5py.File(WAVE_H5) as f:
    EID_ALL = f['event_ids'][:]


In [ ]:
def elbow_threshold(u_vals):
    """
    Elbow method on log10(|u|) — exact match of MATLAB C_SVD_all_15OBS.m logic.
    MATLAB: SVD_num = max(abs(floor(log10(abs(U(:,1)))))) → thresholds = 10^-1 … 10^-SVD_num
    Returns the threshold (as log10 value) at the maximum perpendicular distance.
    """
    u_abs = np.abs(u_vals)
    u_abs = u_abs[u_abs > 0]
    # KEY FIX: abs() of floor — mirrors MATLAB abs(floor(log10(...)))
    sv_num = int(np.max(np.abs(np.floor(np.log10(u_abs)))))
    sv_num = max(sv_num, 2)   # need at least 2 points for elbow
    thresholds = np.array([-i for i in range(1, sv_num + 1)], dtype=float)   # log10
    counts = np.array([np.sum(np.log10(u_abs) > t) for t in thresholds], dtype=float)
    if len(thresholds) < 2:
        return thresholds[0]
    # Fit line from first to last point (polyfit degree 1) — exact MATLAB equivalent
    x, y = thresholds, counts
    p = np.polyfit([x[0], x[-1]], [y[0], y[-1]], 1)  # A, B (y = Ax + B)
    A_c, B_c_neg, C_c = p[0], -1.0, p[1]             # line: A*x - y + C = 0
    dist = np.abs(A_c * x + B_c_neg * y + C_c) / np.sqrt(A_c**2 + B_c_neg**2)
    return thresholds[np.argmax(dist)]


for B_path in sorted(OUT_B.glob(f'B_{YEAR}_*.npy')):
    sta = B_path.stem.replace(f'B_{YEAR}_', '')
    arr = np.load(B_path)   # (M, 3): eid_i, eid_j, polarity_weight

    # Map event_ids → compact indices
    unique_eids = np.unique(arr[:, :2].astype(int))
    eid2idx = {e: k for k, e in enumerate(unique_eids)}
    N = len(unique_eids)

    row = np.array([eid2idx[int(e)] for e in arr[:, 0]])
    col = np.array([eid2idx[int(e)] for e in arr[:, 1]])
    val = arr[:, 2].astype(float)

    # Build symmetric sparse matrix (same as MATLAB: sparseMatrix + sparseMatrix' - diag)
    S = sp.csr_matrix((val, (row, col)), shape=(N, N))
    S = S + S.T - sp.diags(S.diagonal())

    # SVD: take leading singular vector
    try:
        U, sv, Vt = svds(S.astype(float), k=1)
        u1 = U[:, 0]
    except Exception as e:
        print(f'{sta}: SVD failed ({e})')
        continue

    # Elbow threshold on log10|u1|
    thr_log = elbow_threshold(u1)
    thr_val  = 10 ** thr_log

    # SVD result: [event_id, u_value, svd_polarity, agree_count, disagree_count, threshold]
    sign_mat = np.sign(S.toarray())
    pred_mat = np.sign(u1[:, None]) @ np.sign(u1[None, :])
    agree_count    = ((sign_mat == pred_mat) & (sign_mat != 0)).sum(axis=1)
    disagree_count = ((sign_mat != pred_mat) & (sign_mat != 0)).sum(axis=1)

    svd_result = np.column_stack([
        unique_eids,
        u1,
        np.sign(u1),
        agree_count,
        disagree_count,
        np.full(N, thr_log)
    ])
    # Zero out polarities below threshold
    svd_result[np.abs(u1) < thr_val, 2] = 0

    np.save(OUT_C / f'C_{YEAR}_{sta}.npy', svd_result)
    n_valid = (svd_result[:, 2] != 0).sum()
    cc_agree = agree_count.sum() / (agree_count.sum() + disagree_count.sum() + 1e-9)
    print(f'{sta}: {N} events → {n_valid} above threshold '
          f'(thr=10^{thr_log:.1f}={thr_val:.2e})  CC-self-consistency={cc_agree:.3f}')

print('\nStep C done.')
